In [1]:
%load_ext autoreload
%autoreload 2

import os, sys, re
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import time
import PINN
from PINN import reader, models, pl, tl
from PINN.models._density_transport import Density_Transfer, DT_analysis
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchdiffeq import odeint

import matplotlib as mpl
import seaborn as sns
from matplotlib.patches import Patch

import palantir
# import cellrank as cr
import scvelo as scv


os.chdir("/home/wz369/rds/hpc-work/PINN_dynamics")

In [2]:
from importlib import reload
from PINN.models import _density_transport as PDT
reload(PDT)

<module 'PINN.models._density_transport' from '/rds/user/wz369/hpc-work/PINN_dynamics/PINN/models/_density_transport.py'>

In [3]:
sc.settings.set_figure_params(frameon=False, dpi=50, figsize=(3,3))

In [4]:
adata = sc.read_h5ad('data/ery_mk_Aug1.h5ad')

# load trajectory

# evaluate by distance

In [5]:
from torchcfm.optimal_transport import wasserstein

In [6]:
adata

AnnData object with n_obs × n_vars = 22740 × 4814
    obs: 'n_genes', 'n_counts', 'mt_count', 'mt_frac', 'doublet_scores', 'predicted_doublets', 'xist_logn', 'Ygene_logn', 'xist_bin', 'Ygene_bin', 'sex_adata', 'biosample_id', 'cellid', 'RBG', 'SLXid', 'index', '10xsample_description', 'sex_mixed', 'sex_meta', 'mouse_id', 'sortedcells', 'expected_cells_10x', 'cellranger_cellsfound', 'chemistry', 'tom', 'expdate', 'batch', 'timepoint_tx_days', 'start_age', 'sample_id', 'countfile', 'S_score', 'G2M_score', 'phase', 'leiden', 'SLX', 'plate_sorted', 'plate_rearranged', 'well_sorted', 'well_rearranged', 'set_index', 'CI_index', 'mouse_platelabel', 'sort_method', 'sample.name', 'population', 'sex', 'countfolder', 'batch_plate_sorted', 'data_type', 'sex_combined', 'longname', 'anno_man', 'leiden_DM', 'HSCscore', 'nn_HSCscore', 'isroot', 'dpt_pseudotime', 'leiden_orig', 'logk', 'net_prolif', 'log10SR', 'log_density_at_E3', 'log_density_at_E7', 'log_density_at_E12', 'log_density_at_E12_clip', 'l

# load configs, dataset and models

In [7]:
config_dir = "logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense"
result_dir = "results/ery_mk_Aug1_multiscaled_n6"

In [8]:
configs = [file for file in os.listdir(config_dir) if file.endswith(".json")]


In [9]:
# load config for different runs
config_dict = {}
model_dict = {}

for js in os.listdir(config_dir):     
    if not js.endswith(".json"):
        continue

    v = js.split("_")[0][1:] # after V
    config = PINN.ExperimentConfig(os.path.join(config_dir, js))
    config.experiment_config['checkpoint_dir'] = config.experiment_config['checkpoint_dir'].replace("/ssd/users/Wergillius/Project/PINN_dynamics", "/home/wz369/rds/hpc-work/PINN_dynamics")
    config_dict[v] = config

    print(v)
    ckpt = config.find_lastest_ckpt()
    print(ckpt)     # print checkpoint 

    model_dict[v] = PINN.models.pde_params.load_from_checkpoint(ckpt, map_location='cpu')
    print("/n")

0
/rds/user/wz369/hpc-work/PINN_dynamics/logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense/lightning_logs/version_0/checkpoints/epoch=202-val_loss=2.11019945.ckpt
/n
1
/rds/user/wz369/hpc-work/PINN_dynamics/logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense/lightning_logs/version_1/checkpoints/epoch=151-val_loss=0.59822255.ckpt
/n
4
/rds/user/wz369/hpc-work/PINN_dynamics/logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense/lightning_logs/version_4/checkpoints/epoch=125-val_loss=442.87063599.ckpt
/n
2
/rds/user/wz369/hpc-work/PINN_dynamics/logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense/lightning_logs/version_2/checkpoints/epoch=190-val_loss=0.36068392.ckpt
/n
3
/rds/user/wz369/hpc-work/PINN_dynamics/logs/ery_mk_Aug1_multiscaled_n6/pde_params_tsense/lightning_logs/version_3/checkpoints/epoch=200-val_loss=-0.27372548.ckpt
/n


In [10]:
ds_config = config_dict["0"].dataset_config
ds_config['knn_volume'] = eval(config.raw_args['knn_volume'])

DS_full = reader.TwoTimpepoint_AnnDS(adata, split=None, **ds_config)


Dataset : Computing density :
	 `density_funs` not specified, default estimator gaussian kde
Dataset : all cells are used


# test stochastic trajectory

In [46]:
reload(PDT)

<module 'PINN.models._density_transport' from '/rds/user/wz369/hpc-work/PINN_dynamics/PINN/models/_density_transport.py'>

In [47]:
# test model : version 2
device = 'cuda:0'
pde_model = model_dict['2'].to(device)
DT_v2 = PDT.Density_Transfer(pde_model, stochastic=True)

In [48]:
adata.obs['anno_man'].cat.categories

Index(['Ery', 'HSC', 'Int prog', 'Meg'], dtype='object')

In [50]:
stemproj = ['HSC', 'Int prog']
timepoint_key = 'timepoint_tx_days'
celltype_key = 'anno_man'
timepoint_tx_days = sorted(adata.obs[timepoint_key].unique())
t0 = timepoint_tx_days[0]

ad_n6 = DS_full.adata

transport_time = 4
n_interval = 10

HSC_cbs = []
traj_dict = {}

for it,t in tqdm(enumerate(timepoint_tx_days[:6])):

    if (transport_time is not None) and (ds_config['norm_time']==False):
        integrate_time = np.linspace(t/t0, (t+transport_time)/t0 ,n_interval+1) / pde_model.time_scale_factor
    elif (transport_time is not None) and (ds_config['norm_time'] == 'min_minus'):
        integrate_time = np.linspace(t-t0, t + transport_time - t0 ,n_interval+1) / pde_model.time_scale_factor
    else:
        integrate_time = np.linspace(t/t0, timepoint_tx_days[it+1]/t0 ,n_interval+1) / pde_model.time_scale_factor
        
    
    # find cell of time
    start_cell = ad_n6.obs.query(f"`{celltype_key}` in @stemproj & `{timepoint_key}` == @t").index
    start_cell = list(start_cell)
    HSC_cbs.append(start_cell)

    # define initital density and cellstates
    cell_index = [np.where(ad_n6.obs_names == x)[0].item() for x in start_cell]
    u0 = DS_full.u_b[it, cell_index].float().to(device)
    s0 = torch.from_numpy(DS_full.cellstate[cell_index]).float().to(device)

    
    print(integrate_time)
    S_trajectory = [DT_v2.cellstate_drift(s0, integrate_time) for i in range(3)]
    traj_dict[str(t)] =  np.stack(S_trajectory)

0it [00:00, ?it/s]

[0.  0.4 0.8 1.2 1.6 2.  2.4 2.8 3.2 3.6 4. ]


1it [00:00,  1.39it/s]

[4.  4.4 4.8 5.2 5.6 6.  6.4 6.8 7.2 7.6 8. ]


2it [00:14,  8.68s/it]

...

4it [00:45, 12.86s/it]

[46.  46.4 46.8 47.2 47.6 48.  48.4 48.8 49.2 49.6 50. ]


5it [00:59, 13.51s/it]

[73.  73.4 73.8 74.2 74.6 75.  75.4 75.8 76.2 76.6 77. ]


6it [01:14, 12.43s/it]


In [51]:
np.stack(S_trajectory).shape

(3, 11, 1087, 3)

### find the cloest cells

In [53]:
from tqdm import tqdm


sim_nn_dict = {}
for day, Trajs in traj_dict.items():

    nn = []
    for r in tqdm(range(3)): # repeat  

        nn_steps = []
        for step in range(1,11):  # step
            nns = tl.assign_nearest_cell(Trajs[r,step], ad_n6, cellstate_key=ds_config['cellstate_key'], n_neighbors=5)
            nn_steps.append(nns)
        nn.append(nn_steps)

    sim_nn_dict[day] = nn 

100%|██████████| 3/3 [00:01<00:00,  2.10it/s]


In [54]:
# by cell type
def summarize_cell_proportions(df, celltype_list):
    """
    Summarizes the proportion of cell types mapped from neighbors for each cell

    Inputs:
    df : Input DataFrame where each column represents a cell and each row a sample.
    celltype_list : List of cell types to include in the output as columns.
    """
    # Compute normalized value counts for each cell
    proportions = df.apply(lambda col: col.value_counts(normalize=True))
    proportions = proportions.fillna(0).T # cell as index

    # Reindex columns to match the given celltype_list, filling missing with 0
    proportions = proportions.reindex(columns=celltype_list, fill_value=0)

    return proportions

In [59]:
celltype_list = ad_n6.obs['anno_man'].cat.categories

traj_ct_dict = {}

for day, Trajs in traj_dict.items():
    ct_df_steps = []
    for step in tqdm(range(1,11)):  # step

        # prop_step = pd.DataFrame()
        ncell = Trajs[0,step].shape[0]
        prop_step = np.zeros((ncell,len(celltype_list)))
        
        for r in range(3):
            cts = tl.assign_nearest_cell(Trajs[r,step], ad_n6, cellstate_key=ds_config['cellstate_key'], n_neighbors=5, annotation='anno_man')
            props = summarize_cell_proportions(cts, celltype_list)
            prop_step += props.values
        
        prop_step = pd.DataFrame(prop_step, columns=celltype_list) /3 
        ct_df_steps.append(prop_step)

    traj_ct_dict[day] = ct_df_steps

100%|██████████| 10/10 [00:13<00:00,  1.37s/it]


In [60]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['3']])

,Ery,HSC,Int prog,Meg
0,0.000913,0.716438,0.255708,0.026941
0,0.002740,0.698174,0.262100,0.036986
0,0.004110,0.673516,0.280365,0.042009
0,0.006393,0.641553,0.309132,0.042922
0,0.010959,0.606849,0.331507,0.050685
0,0.012785,0.569863,0.362100,0.055251
0,0.026027,0.504566,0.415525,0.053881
0,0.042922,0.422831,0.471233,0.063014
0,0.053425,0.358447,0.517352,0.070776
0,0.068493,0.301370,0.553881,0.076256


In [61]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['7']])

,Ery,HSC,Int prog,Meg
0,0.014848,0.579697,0.390909,0.014545
0,0.016061,0.561212,0.387879,0.034848
0,0.016061,0.546364,0.389394,0.048182
0,0.013333,0.520606,0.406364,0.059697
0,0.015758,0.485152,0.425152,0.073939
0,0.013333,0.426061,0.479394,0.081212
0,0.014545,0.352727,0.538182,0.094545
0,0.013030,0.272424,0.600000,0.114545
0,0.011818,0.196667,0.661818,0.129697
0,0.012121,0.134242,0.708485,0.145152


In [62]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['12']])

,Ery,HSC,Int prog,Meg
0,0.007356,0.418851,0.527816,0.045977
0,0.007816,0.395402,0.531494,0.065287
0,0.005977,0.366897,0.547126,0.080000
0,0.005517,0.330115,0.562299,0.102069
0,0.005517,0.296092,0.592644,0.105747
0,0.005977,0.254713,0.616092,0.123218
0,0.006437,0.231724,0.624828,0.137011
0,0.006437,0.179310,0.656092,0.158161
0,0.006897,0.153103,0.668506,0.171494
0,0.006897,0.104828,0.689655,0.198621


In [63]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['27']])

,Ery,HSC,Int prog,Meg
0,0.002175,0.466165,0.493936,0.037725
0,0.001004,0.452363,0.495859,0.050774
0,0.000836,0.434546,0.502133,0.062484
0,0.000836,0.422083,0.503722,0.073358
0,0.001171,0.404182,0.503806,0.090841
0,0.000502,0.391217,0.497281,0.111000
0,0.000586,0.376997,0.490757,0.131660
0,0.000502,0.357591,0.485404,0.156504
0,0.000335,0.332999,0.479297,0.187369
0,0.000502,0.287997,0.493015,0.218486


In [66]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['49']])

,Ery,HSC,Int prog,Meg
0,0.002416,0.448105,0.509720,0.039758
0,0.001647,0.437562,0.513234,0.047556
0,0.002306,0.424272,0.518616,0.054805
0,0.002306,0.413948,0.518396,0.065349
0,0.002416,0.408786,0.509940,0.078858
0,0.001977,0.388138,0.515102,0.094783
0,0.002087,0.379572,0.505107,0.113234
0,0.002197,0.370895,0.492147,0.134761
0,0.002416,0.348600,0.492916,0.156068
0,0.002526,0.328940,0.483800,0.184734


In [67]:
traj_ct_dict.keys()

dict_keys(['3', '7', '12', '27', '49', '76'])

In [68]:
pd.concat([pd.DataFrame(df.mean(axis=0)).T for df in traj_ct_dict['76']])

,Ery,HSC,Int prog,Meg
0,0.003680,0.450721,0.509230,0.036369
0,0.001595,0.440662,0.515057,0.042686
0,0.001288,0.430359,0.517755,0.050598
0,0.001165,0.422938,0.519902,0.055995
0,0.000920,0.414597,0.520025,0.064459
0,0.000797,0.407053,0.513953,0.078197
0,0.000675,0.399693,0.506838,0.092794
0,0.000859,0.392027,0.497700,0.109414
0,0.000859,0.381785,0.491138,0.126219
0,0.000981,0.373014,0.478074,0.147930


In [20]:
# TODO: calculate pseudotime
# TODO: calculate distance